In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import random
import matplotlib.pyplot as plt
from torch.utils.data import Subset
import numpy as np
import math
from tqdm import tqdm

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED=42

set_seed(SEED)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] Available device: {device}")

In [ ]:

BASE_DIR=os.path.dirname(os.getcwd())
DATASET="Flowers102"
METHOD="KCenter_feature_vectors"
METHODS=["FewMedoids_feature_vectors","Herding_feature_vectors","KCenter_feature_vectors","Random"]
MULTI_SEED_METHODS={"KCenter_feature_vectors"}

SPLIT=0.2
SPLIT_DIR=os.path.join(BASE_DIR,"split",DATASET,f"seed_{SEED}",f"split_{SPLIT}")

K=[1,2,4,8]
SEEDS=[0,1,2,3,4]

DTYPE="float32"
DATA=os.path.join(BASE_DIR,"data")

MODEL_PATH=os.path.join(BASE_DIR,
                        "model",
                        "ViT_B_16",
                        "Flowers102",
                        "seed_42",
                        "split_0.2",
                        "val_acc=96.08%_val_loss=0.2194_lr=0.0001_wd=0.05_ep=100_bs=128_opt=AdamW_sch=CosineAnnealingLR",
                        "model.pth")

MODEL_NAME="ViT_B_16"
IMG_SIZE=224
BATCH_SIZE=128

if DATASET=="CIFAR10":
    NUM_CLASSES=10
elif DATASET=="CIFAR100":
    NUM_CLASSES=100
elif DATASET=="Food101":
    NUM_CLASSES=101
elif DATASET=="Flowers102":
    NUM_CLASSES=102

MATRICES_DIR=os.path.join(DATASET,"matrices")
MATRICES_INDICES_DIR=os.path.join(DATASET,"matrices_indices")
METHODS_DIR=os.path.join(DATASET,f"methods_{MODEL_NAME}")
RANDOM_DIR=os.path.join(DATASET,"Random")
FEATURE_VECTORS_DIR=os.path.join(DATASET,f"feature_vectors_{MODEL_NAME}")

BUILD_MATRICES=False
RUN_SELECTIONS=True
FEATURE_VECTORS=False

for path in [DATASET,MATRICES_DIR,MATRICES_INDICES_DIR,METHODS_DIR,RANDOM_DIR,FEATURE_VECTORS_DIR]:
    
    os.makedirs(path,exist_ok=True)

for method_folder in METHODS:

    if method_folder=="Random":

        method_path=os.path.join(RANDOM_DIR)

        for seed in SEEDS:

            seed_path=os.path.join(method_path,f"seed_{seed}")
            os.makedirs(seed_path,exist_ok=True)

            split_path=os.path.join(seed_path,f"split_{SPLIT}")
            os.makedirs(split_path,exist_ok=True)

            for k in K:

                k_path=os.path.join(split_path,f"k_{k}")
                os.makedirs(k_path,exist_ok=True)

                plots_path=os.path.join(k_path,"plots")
                os.makedirs(plots_path,exist_ok=True)

    elif method_folder in MULTI_SEED_METHODS:

        method_path=os.path.join(METHODS_DIR, method_folder)
        os.makedirs(method_path,exist_ok=True)

        for seed in SEEDS:

            seed_path=os.path.join(method_path,f"seed_{seed}")
            os.makedirs(seed_path,exist_ok=True)

            split_path=os.path.join(seed_path,f"split_{SPLIT}")
            os.makedirs(split_path,exist_ok=True)

            for k in K:

                k_path=os.path.join(split_path,f"k_{k}")
                os.makedirs(k_path,exist_ok=True)

                plots_path=os.path.join(k_path,"plots")
                os.makedirs(plots_path,exist_ok=True)

    else:

        method_path=os.path.join(METHODS_DIR, method_folder)
        os.makedirs(method_path,exist_ok=True)

        seed_path=os.path.join(method_path,f"seed_{SEED}")
        split_path=os.path.join(seed_path,f"split_{SPLIT}")

        os.makedirs(split_path,exist_ok=True)

        for k in K:

            k_path=os.path.join(split_path,f"k_{k}")
            os.makedirs(k_path,exist_ok=True)

            plots_path=os.path.join(k_path,"plots")
            os.makedirs(plots_path,exist_ok=True)


In [ ]:

if METHOD not in METHODS:
    raise ValueError(f"Unsupported method: {METHOD}")



In [ ]:
def load_dataset(dataset_name):

    if dataset_name=="CIFAR10":
        dataset=torchvision.datasets.CIFAR10(root=DATA,train=True,download=True,transform=None)
    
    elif dataset_name=="CIFAR100":
        dataset=torchvision.datasets.CIFAR100(root=DATA,train=True,download=True,transform=None)

    elif dataset_name=="Food101":
        dataset=torchvision.datasets.Food101(root=DATA,split="train",download=True,transform=None)

    elif dataset_name=="Flowers102":
        dataset=torchvision.datasets.Flowers102(root=DATA,split="train",download=True,transform=None)

    else:
        raise ValueError(f"Dataset {dataset_name} is not currently supported")

    train_indices=np.load(os.path.join(SPLIT_DIR,"train_indices.npy"))
    dataset=Subset(dataset,train_indices)

    underlying=dataset.dataset if isinstance(dataset, Subset) else dataset
    if hasattr(underlying,"class_to_idx"):
        class_to_idx=underlying.class_to_idx
    else:
        class_to_idx={str(i):i for i in range(NUM_CLASSES)}
    idx_to_class={v:k for k,v in class_to_idx.items()}

    print(f"\n[OK] Loaded dataset: {dataset_name}")
    print(f"Number of classes: {len(class_to_idx)}")
    print("Classes:",list(class_to_idx.keys())[:])

    return dataset,class_to_idx,idx_to_class


In [ ]:
dataset_obj,class_to_idx,idx_to_class=load_dataset(DATASET)

In [ ]:
def build_class_matrices(dataset_obj, class_to_idx, idx_to_class):

    print(f"Building class matrices for dataset: {DATASET}:\n")

    dataset=dataset_obj
    num_classes=len(class_to_idx)

    if DATASET in ["Flowers102","Food101"]:
        to_tensor=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor()])
    else:
        to_tensor=transforms.ToTensor()

    per_class_tensors={i: [] for i in range(num_classes)}
    per_class_indices={i: [] for i in range(num_classes)}

    for idx in range(len(dataset)):
        img,label=dataset[idx]

        if not isinstance(img, torch.Tensor):
            img=to_tensor(img)

        per_class_tensors[label].append(img)
        per_class_indices[label].append(idx)

    print("Images grouped by class. Starting matrix creation:\n")

    for class_id,class_name in idx_to_class.items():

        images=per_class_tensors[class_id]
        indices=np.array(per_class_indices[class_id],dtype=np.int64)

        if len(images)==0:
            print(f"[WARN] No samples found for class '{class_name}'!")
            continue

        class_matrix=torch.stack(images,dim=0).numpy()
        class_matrix=class_matrix.reshape(class_matrix.shape[0],-1)
        class_matrix=class_matrix.astype(DTYPE)

        np.save(os.path.join(MATRICES_DIR,f"class_{class_id}_{class_name}.npy"),class_matrix)
        np.save(os.path.join(MATRICES_INDICES_DIR,f"class_{class_id}_{class_name}_indices.npy"),indices)

        print(f"[OK] Saved class '{class_name}' | shape: {class_matrix.shape}")

        del class_matrix
        per_class_tensors[class_id]=[]


    print(f"\nAll class matrices saved in: {MATRICES_DIR}\n")
    print(f"All class indices saved in: {MATRICES_INDICES_DIR}\n")

In [ ]:
if BUILD_MATRICES:
    
    build_class_matrices(dataset_obj,class_to_idx,idx_to_class)

In [ ]:
def build_feature_vectors(dataset_obj, class_to_idx, idx_to_class):

    print(f"Building feature vectors for dataset: {DATASET}:\n")

    num_classes=len(class_to_idx)

    transform=transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
    ])


    model_builders={
        "ResNet34": models.resnet34,
        "ViT_B_16": models.vit_b_16
    }

    if MODEL_NAME not in model_builders:
        raise ValueError(f"Unsupported architecture: {MODEL_NAME}")

    model=model_builders[MODEL_NAME](weights=None)

    if hasattr(model,"heads"):
        in_features=model.heads.head.in_features
        model.heads.head=nn.Linear(in_features,NUM_CLASSES)
    elif hasattr(model,"fc"):
        in_features=model.fc.in_features
        model.fc=nn.Linear(in_features,NUM_CLASSES)
    elif hasattr(model,"classifier"):
        if isinstance(model.classifier,nn.Sequential):
            in_features=model.classifier[-1].in_features
            model.classifier[-1]=nn.Linear(in_features,NUM_CLASSES)
        else:
            in_features=model.classifier.in_features
            model.classifier=nn.Linear(in_features,NUM_CLASSES)

    model.load_state_dict(torch.load(MODEL_PATH,map_location=device,weights_only=True))
    model=model.to(device)
    model.eval()

    for param in model.parameters():
        param.requires_grad=False

    if hasattr(model,"heads"):
        model.heads.head=nn.Identity()
    elif hasattr(model,"fc"):
        model.fc=nn.Identity()
    elif hasattr(model,"classifier"):
        model.classifier=nn.Identity()

    print(f"[OK] Model loaded from: {MODEL_PATH}")
    print(f"[OK] Architecture: {MODEL_NAME} | Classes: {NUM_CLASSES}\n")
    print(f"Extracting feature vectors using {MODEL_NAME}...\n")

    per_class_indices={i: [] for i in range(num_classes)}

    for idx in range(len(dataset_obj)):
        _,label=dataset_obj[idx]
        per_class_indices[label].append(idx)


    for class_id,class_name in idx_to_class.items():

        indices=per_class_indices[class_id]

        if len(indices)==0:
            print(f"[WARN] No samples found for class '{class_name}'!")
            continue

        class_features=[]
        num_batches=math.ceil(len(indices)/BATCH_SIZE)

        for start in tqdm(range(0,len(indices),BATCH_SIZE),desc=f"[{class_id+1}/{num_classes}] {class_name}",total=num_batches):

            batch_indices=indices[start:start+BATCH_SIZE]
            batch_images=[]

            for i in batch_indices:
                img,_=dataset_obj[i]
                img=transform(img)
                batch_images.append(img)

            batch_tensor=torch.stack(batch_images,dim=0).to(device)

            with torch.no_grad():
                features=model(batch_tensor)

            class_features.append(features.cpu().numpy())

        class_features=np.concatenate(class_features,axis=0).astype(DTYPE)

        np.save(os.path.join(FEATURE_VECTORS_DIR,f"class_{class_id}_{class_name}.npy"),class_features)

        print(f"[OK] Saved class '{class_name}' | features shape: {class_features.shape} | images: {len(indices)}\n")

        del class_features

    print(f"All feature vectors saved in: {FEATURE_VECTORS_DIR}")
    print(f"Indices are shared with: {MATRICES_INDICES_DIR}\n")

In [ ]:
if FEATURE_VECTORS:

    build_feature_vectors(dataset_obj,class_to_idx,idx_to_class)

In [ ]:
# Compute a centrality score for each point by averaging its distance to all other points.
# This ranks points by how representative they are: lower average distance means more central.
# The first element corresponds to the global medoid, while others follow in decreasing centrality.

def FewMedoids_feature_vectors(X):
    X = X.astype(np.float32)

    rank = []
    for i in range(X.shape[0]):
        dists = np.linalg.norm(X - X[i], axis=1)
        rank.append((i, float(dists.mean())))

    rank.sort(key=lambda x: x[1])
    return rank

# k-Center-Greedy with random s0 initialization, following the implementations from:
# - "Active Learning for Convolutional Neural Networks: A Core-Set Approach"
# - Google active-learning reference implementation
#   https://github.com/google/active-learning/blob/master/sampling_methods/kcenter_greedy.py
# - DeepCore k-Center-Greedy implementation
#   https://github.com/PatrickZH/DeepCore/blob/main/deepcore/methods/kcentergreedy.py
#
# As in these implementations, the initial point (first_idx / s0) is selected uniformly at random
# (therefore the result depends on the random seed). After that, points are added
# iteratively by selecting the sample that is farthest from its nearest already
# selected center.
#
# In other words, at each step we choose the point with the largest minimum distance
# to the current selected set, which greedily reduces the worst-case covering distance
# over the dataset.
#
# Here s0 is included as the first selected center, so the returned ranking contains
# exactly k exemplars in total, with s0 counted inside that budget.

def KCenter_feature_vectors(X, first_idx):
    k = 128
    X = X.astype(np.float32)

    first = first_idx

    selected = [first]
    used = np.zeros(X.shape[0], dtype=bool)
    used[first] = True

    min_dists = np.linalg.norm(X - X[first], axis=1)

    target_size = min(k, X.shape[0])

    while len(selected) < target_size:
        min_dists[used] = -1.0
        nxt = int(np.argmax(min_dists))
        selected.append(nxt)
        used[nxt] = True

        new_dists = np.linalg.norm(X - X[nxt], axis=1)
        min_dists = np.minimum(min_dists, new_dists)

    return [(idx, rank) for rank, idx in enumerate(selected)]


# Herding-based exemplar selection, following the implementations from:
# - Welling (2009)
# - iCaRL: Rebuffi et al., CVPR 2017
#   https://github.com/srebuffi/iCaRL/blob/master/iCaRL-TheanoLasagne/main_cifar_100_theano.py
# - DeepCore herding implementation
#   https://github.com/PatrickZH/DeepCore/blob/main/deepcore/methods/herding.py
#
# Here we follow the same idea used in those implementations:
# feature vectors are first L2-normalized, then exemplars are selected greedily with
#
#   x_t = argmax_x <w_t, phi(x)>
#   w_{t+1} = w_t + mu - phi(x)
#
# Because ||phi(x)|| = 1 after L2 normalization, maximizing the dot product
# <w_t, phi(x)> is equivalent to minimizing the squared Euclidean distance
# between w_t and phi(x):
#
#   ||w_t - phi(x)||^2
#     = ||w_t||^2 - 2<w_t, phi(x)> + ||phi(x)||^2
#     = ||w_t||^2 - 2<w_t, phi(x)> + 1
#
# Since ||w_t||^2 and 1 do not depend on x, minimizing the distance is
# equivalent to maximizing <w_t, phi(x)>.

def Herding_feature_vectors(X):
    k = 128
    X = X.astype(np.float32)

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / norms

    mu = X.mean(axis=0)

    used = np.zeros(X.shape[0], dtype=bool)
    w_t = mu.copy()

    rank = []

    for t in range(min(k, X.shape[0])):
        dots = X @ w_t
        dots[used] = -np.inf

        idx = int(np.argmax(dots))

        rank.append((idx, t))

        used[idx] = True
        w_t = w_t + mu - X[idx]

    return rank


In [ ]:

def run_selection(idx_to_class):

    print(f"Running selection for dataset: {DATASET} | method: {METHOD}\n")

    method_map={
        "FewMedoids_feature_vectors": FewMedoids_feature_vectors,
        "KCenter_feature_vectors": KCenter_feature_vectors,
        "Herding_feature_vectors": Herding_feature_vectors
    }

    if METHOD=="Random":

        for seed in SEEDS:

            set_seed(seed)

            for class_id, class_name in idx_to_class.items():

                index_path = os.path.join(MATRICES_INDICES_DIR,f"class_{class_id}_{class_name}_indices.npy")

                if not os.path.exists(index_path):
                    print(f"[WARN] Missing files for class '{class_name}'!")
                    continue

                global_indices=np.load(index_path)
                order_full=np.random.permutation(len(global_indices))

                for k in K:

                    top_local=order_full[:k]
                    top_global=global_indices[top_local]

                    out_path=os.path.join(RANDOM_DIR,f"seed_{seed}",f"split_{SPLIT}",f"k_{k}")
                    os.makedirs(out_path, exist_ok=True)

                    np.save(os.path.join(out_path, f"class_{class_id}_{class_name}_indices.npy"),top_global.astype(np.int64))

                print(f"[OK] Saved random selections for class '{class_name}' | seed={seed}")

    elif METHOD in MULTI_SEED_METHODS:

        selector=method_map[METHOD]

        for seed in SEEDS:

            set_seed(seed)

            for class_id, class_name in idx_to_class.items():

                feature_path = os.path.join(FEATURE_VECTORS_DIR,f"class_{class_id}_{class_name}.npy")
                index_path = os.path.join(MATRICES_INDICES_DIR,f"class_{class_id}_{class_name}_indices.npy")

                if not os.path.exists(feature_path) or not os.path.exists(index_path):
                    print(f"[WARN] Missing files for class '{class_name}'!")
                    continue

                class_features=np.load(feature_path)
                global_indices=np.load(index_path)

                if class_features.size==0:
                    print(f"[WARN] Empty feature vectors for class '{class_name}'!")
                    continue

                random_k1_path=os.path.join(RANDOM_DIR,f"seed_{seed}",f"split_{SPLIT}","k_1",f"class_{class_id}_{class_name}_indices.npy")
                random_global=int(np.load(random_k1_path)[0])
                first_idx=int(np.where(global_indices==random_global)[0][0])

                rank=selector(class_features,first_idx=first_idx)
                order_full=np.array([idx for idx,_ in rank])

                base_path=os.path.join(METHODS_DIR,METHOD,f"seed_{seed}",f"split_{SPLIT}")
                os.makedirs(base_path,exist_ok=True)

                for k in K:

                    top_local=order_full[:k]
                    top_global=global_indices[top_local]

                    out_path=os.path.join(base_path,f"k_{k}")
                    os.makedirs(out_path,exist_ok=True)

                    np.save(os.path.join(out_path,f"class_{class_id}_{class_name}_indices.npy"),top_global.astype(np.int64))

                print(f"[OK] Saved selections for class '{class_name}' using {METHOD} | seed={seed}")

    else:
        selector=method_map[METHOD]

        for class_id, class_name in idx_to_class.items():

            feature_path = os.path.join(FEATURE_VECTORS_DIR,f"class_{class_id}_{class_name}.npy")
            index_path = os.path.join(MATRICES_INDICES_DIR,f"class_{class_id}_{class_name}_indices.npy")

            if not os.path.exists(feature_path) or not os.path.exists(index_path):

                print(f"[WARN] Missing files for class '{class_name}'!")
                continue

            class_features=np.load(feature_path)
            global_indices=np.load(index_path)

            if class_features.size==0:

                print(f"[WARN] Empty feature vectors for class '{class_name}'!")
                continue

            print(f"\nProcessing class: {class_name} | shape: {class_features.shape}:\n")

            rank=selector(class_features)
            order_full=np.array([idx for idx,_ in rank])

            base_path=os.path.join(METHODS_DIR,METHOD,f"seed_{SEED}",f"split_{SPLIT}")
            os.makedirs(base_path,exist_ok=True)

            for k in K:

                top_local=order_full[:k]
                top_global=global_indices[top_local]

                out_path=os.path.join(base_path,f"k_{k}")
                os.makedirs(out_path,exist_ok=True)

                np.save(os.path.join(out_path,f"class_{class_id}_{class_name}_indices.npy"),top_global.astype(np.int64))

                print(f"[OK] Saved selections for class '{class_name}' using {METHOD}")

                print("\nAll selections saved successfully.\n")

In [ ]:
if RUN_SELECTIONS:
    
    run_selection(idx_to_class)

In [ ]:

def plot_selected_examples(dataset_obj,class_id,idx_to_class,k,save_plot=False,seed=None):

    dataset=dataset_obj
    class_name=idx_to_class[class_id]

    if METHOD=="Random":

        method_path=RANDOM_DIR

        if seed is None:
            print("[WARN] Missing 'seed' parameter for Random method!")
            return
        
        indices_path=os.path.join(method_path,f"seed_{seed}",f"split_{SPLIT}",f"k_{k}",f"class_{class_id}_{class_name}_indices.npy")

    elif METHOD in MULTI_SEED_METHODS:

        method_path=os.path.join(METHODS_DIR,METHOD)

        if seed is None:
            print(f"[WARN] Missing 'seed' parameter for {METHOD} method!")
            return

        indices_path=os.path.join(method_path,f"seed_{seed}",f"split_{SPLIT}",f"k_{k}",f"class_{class_id}_{class_name}_indices.npy")

    else:

        method_path=os.path.join(METHODS_DIR,METHOD)
        indices_path=os.path.join(method_path,f"seed_{SEED}",f"split_{SPLIT}",f"k_{k}",f"class_{class_id}_{class_name}_indices.npy")

    if not os.path.exists(indices_path):
        print(f"[WARN] Missing indices file: {indices_path}")
        return

    selected_indices=np.load(indices_path)
    print(f"Plotting {k} examples for class {class_name} | ID: {class_id} using {METHOD}")

    cols=math.ceil(math.sqrt(k))
    rows=math.ceil(k/cols)

    fig,axes=plt.subplots(rows,cols,figsize=(cols*3.0,rows*3.0))
    
    if isinstance(axes, np.ndarray):
        
        axes=axes.flatten()
    else:
        axes=[axes]
    
    for i,ax in enumerate(axes[:min(len(selected_indices),rows*cols)]):

        img,label=dataset[selected_indices[i]]

        if isinstance(img, torch.Tensor):
            img_np=img.detach().cpu().permute(1,2,0).numpy()
            
        else:
            img_np=np.array(img)  

        ax.imshow(img_np)

        ax.set_title(f"{class_name} #{selected_indices[i]}",fontsize=15,pad=6)
        ax.axis("off")

    for ax in axes[len(selected_indices):]:
        ax.axis("off")

    plt.suptitle(f"{DATASET} | {METHOD} | Class: {class_id} | top-{k}",fontsize=36,y=0.92)

    plt.subplots_adjust(top=0.90)
    plt.tight_layout(rect=[0,0,1,0.88])

    if save_plot:

        if METHOD=="Random":
            plots_dir=os.path.join(method_path,f"seed_{seed}",f"split_{SPLIT}",f"k_{k}","plots")
  
        elif METHOD in MULTI_SEED_METHODS:
            plots_dir=os.path.join(method_path,f"seed_{seed}",f"split_{SPLIT}",f"k_{k}","plots")

        else:
            plots_dir=os.path.join(method_path,f"seed_{SEED}",f"split_{SPLIT}",f"k_{k}","plots")

        os.makedirs(plots_dir, exist_ok=True)

        filename=f"class_{class_id}.pdf"
        out_path=os.path.join(plots_dir, filename)

        plt.savefig(out_path,bbox_inches="tight",dpi=300)
        print(f"[OK] Saved plot: {out_path}\n")
        plt.close(fig)

    else:
        
        plt.show()
        plt.close(fig)


In [ ]:
METHOD="KCenter_feature_vectors"
plot_selected_examples(dataset_obj=dataset_obj,class_id=2,idx_to_class=idx_to_class,k=8,save_plot=False,seed=4)
